In [2]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

file_path = ("result_retail.csv")

df = pd.read_csv(
    file_path,
    parse_dates=["InvoiceDate"]
)

df.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom


## Dataset Summary

The purpose of this section is to obtain an overview of the merged dataset before applying any cleaning operations.

In [3]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1033036.00,1033036,1033036.00,797885.00
mean,10.08,2011-01-03 14:30:35.429549824,4.61,15313.06
min,-80995.00,2009-12-01 07:45:00,-53594.36,12346.00
25%,1.00,2010-07-05 11:38:00,1.25,13964.00
50%,3.00,2010-12-09 13:34:00,2.10,15228.00
75%,10.00,2011-07-27 13:17:00,4.15,16788.00
max,80995.00,2011-12-09 12:50:00,38970.00,18287.00
std,175.20,NaN,122.40,1696.47


In [4]:
missing_data = pd.DataFrame({"Missing_values": df.isnull().sum(),
                            "Percentage" : (df.isnull().sum()*100/len(df)).round(2)})
missing_data
## Which missing values affect downstream analyses?

,Missing_values,Percentage
Invoice,0,0.00
StockCode,0,0.00
Description,4275,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,235151,22.76
Country,0,0.00


In [5]:
print(df.duplicated().sum())

0


In [6]:
cancelled = df["Invoice"].astype(str).str.startswith("C")

print(cancelled.sum())

19104


### Observation

Cancelled invoices are identified by invoice numbers beginning with "C". These transactions most likely represent returned or cancelled orders rather than data entry errors.

They should not be included in revenue calculations but may be useful when analyzing return behavior.

In [7]:
df[cancelled].head(30)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.00,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.00,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.00,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.00,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.00,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.00,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.00,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.00,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.00,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.00,United Kingdom


In [8]:
(df["Quantity"] < 0).sum()
df[df["Quantity"] < 0].head(30)
## Are all negative quantities linked to cancelled invoices?

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.00,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.00,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.00,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.00,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.00,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.00,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.00,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.00,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.00,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.00,United Kingdom


In [9]:
print((df["Price"] <= 0).sum())
print(df[df["Price"] <= 0])

6019
        Invoice StockCode                   Description  Quantity  \
263      489464     21733                  85123a mixed       -96   
283      489463     71477                         short      -240   
284      489467    85123A                   21733 mixed      -192   
462      489521     21646                           NaN       -50   
3077     489655     20683                           NaN       -44   
...         ...       ...                           ...       ...   
1028147  581234     72817                           NaN        27   
1029653  581406    46000M  POLYESTER FILLER PAD 45x45cm       240   
1029654  581406    46000S  POLYESTER FILLER PAD 40x40cm       300   
1029703  581408     85175                           NaN        20   
1030065  581422     23169                       smashed      -235   

                InvoiceDate  Price  Customer ID         Country  
263     2009-12-01 10:52:00   0.00          NaN  United Kingdom  
283     2009-12-01 10:52:00   0.00

In [10]:
print(df["Description"].isnull().sum())
df[df["Description"].isnull()]
## Are they linked to missing customer IDs?

4275


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
462,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
3077,489655,20683,NaN,-44,2009-12-01 17:26:00,0.00,NaN,United Kingdom
3124,489659,21350,NaN,230,2009-12-01 17:39:00,0.00,NaN,United Kingdom
3687,489781,84292,NaN,17,2009-12-02 11:45:00,0.00,NaN,United Kingdom
4233,489806,18010,NaN,-770,2009-12-02 12:42:00,0.00,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
1026488,581199,84581,NaN,-2,2011-12-07 18:26:00,0.00,NaN,United Kingdom
1026492,581203,23406,NaN,15,2011-12-07 18:31:00,0.00,NaN,United Kingdom
1026498,581209,21620,NaN,6,2011-12-07 18:35:00,0.00,NaN,United Kingdom
1028147,581234,72817,NaN,27,2011-12-08 10:33:00,0.00,NaN,United Kingdom


In [11]:
print(df["Customer ID"].isnull().sum())
df[df["Customer ID"].isnull()].head(20)

235151


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
462,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
569,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
570,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1027,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1028,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1029,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1030,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom


In [12]:
quality = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Missing Customer IDs",
        "Missing Descriptions",
        "Cancelled Invoices",
        "Negative Quantities",
        "Zero/Negative Prices"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df["Customer ID"].isnull().sum(),
        df["Description"].isnull().sum(),
        cancelled.sum(),
        (df["Quantity"] < 0).sum(),
        (df["Price"] <= 0).sum()
    ]
})

quality

,Metric,Value
0,Rows,1033036
1,Columns,8
2,Duplicate Rows,0
3,Missing Customer IDs,235151
4,Missing Descriptions,4275
5,Cancelled Invoices,19104
6,Negative Quantities,22496
7,Zero/Negative Prices,6019


## Summary

The following data quality issues were investigated:

- Missing Customer IDs
- Missing Descriptions
- Duplicate Records
- Cancelled Invoices
- Negative Quantities
- Zero/Negative Prices

The next section will apply cleaning decisions based on these investigations instead of generic preprocessing rules.

In [13]:
missing_desc = df[df["Description"].isnull()]

missing_desc.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
462,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
3077,489655,20683,NaN,-44,2009-12-01 17:26:00,0.00,NaN,United Kingdom
3124,489659,21350,NaN,230,2009-12-01 17:39:00,0.00,NaN,United Kingdom
3687,489781,84292,NaN,17,2009-12-02 11:45:00,0.00,NaN,United Kingdom
4233,489806,18010,NaN,-770,2009-12-02 12:42:00,0.00,NaN,United Kingdom
4499,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.00,NaN,United Kingdom
6299,489882,35751C,NaN,12,2009-12-02 16:22:00,0.00,NaN,United Kingdom
6476,489898,79323G,NaN,954,2009-12-03 09:40:00,0.00,NaN,United Kingdom
6497,489901,21098,NaN,-200,2009-12-03 09:47:00,0.00,NaN,United Kingdom
6502,489903,21166,NaN,48,2009-12-03 09:57:00,0.00,NaN,United Kingdom


In [14]:
missing_desc[
    ["Invoice","StockCode","Customer ID","Quantity","Price"]
].head(20)

,Invoice,StockCode,Customer ID,Quantity,Price
462,489521,21646,NaN,-50,0.00
3077,489655,20683,NaN,-44,0.00
3124,489659,21350,NaN,230,0.00
3687,489781,84292,NaN,17,0.00
4233,489806,18010,NaN,-770,0.00
4499,489821,85049G,NaN,-240,0.00
6299,489882,35751C,NaN,12,0.00
6476,489898,79323G,NaN,954,0.00
6497,489901,21098,NaN,-200,0.00
6502,489903,21166,NaN,48,0.00


In [15]:
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom
...,...,...,...,...,...,...,...,...
1033031,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.00,France
1033032,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.00,France
1033033,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.00,France
1033034,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.00,France


In [16]:
df = df.dropna(subset=["Description"])

### Cleaning Decision – Missing Descriptions

Records with missing product descriptions were removed because they cannot contribute to product-level analytics such as Market Basket Analysis, Product Recommendation, or Product Performance Analysis.

All removals were documented to preserve transparency in the data cleaning pipeline.

In [17]:
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom
...,...,...,...,...,...,...,...,...
1033031,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.00,France
1033032,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.00,France
1033033,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.00,France
1033034,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.00,France


Without a product description:

Market Basket Analysis becomes unreliable.
Product Recommendation is affected.
Product-level EDA becomes incomplete.

These records contribute little analytical value.

In [18]:
## Datasets of cancelled items 
returns_df = df[
    df["Invoice"].astype(str).str.startswith("C")
].copy()

In [19]:
df = df[
    ~df["Invoice"].astype(str).str.startswith("C")
].copy()

In [20]:
print("Remaining Rows:" ,len(df))

Remaining Rows: 1009657


In [ ]:
## Validation and a assurity check
df["Invoice"].astype(str).str.startswith("C").sum()

np.int64(0)

## Cleaning Decision – Cancelled Transactions

Cancelled invoices were identified using invoice numbers beginning with the letter **'C'**.

These records represent returned or cancelled orders rather than completed sales transactions.

Instead of deleting them permanently, they were preserved in a separate dataset (`returns_data.csv`) for future return-rate and product return analysis.

They were removed from the primary analytical dataset to ensure that revenue, customer behavior, forecasting, and product performance analyses are based only on completed sales.

In [25]:
negative_qty = df[df['Quantity']<0]
negative_qty.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
3125,489660,35956,lost,-1043,2009-12-01 17:43:00,0.00,NaN,United Kingdom
3131,489663,35605A,damages,-117,2009-12-01 18:02:00,0.00,NaN,United Kingdom
4471,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.00,NaN,United Kingdom
6477,489899,79323GR,sold as gold,-954,2009-12-03 09:41:00,0.00,NaN,United Kingdom
6832,490007,84347,21494,-720,2009-12-03 12:09:00,0.00,NaN,United Kingdom
9216,490130,21493,lost?,-600,2009-12-03 18:28:00,0.00,NaN,United Kingdom
17226,490765,21450,damaged,-31,2009-12-08 10:59:00,0.00,NaN,United Kingdom


In [24]:
negative_qty.tail(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1026493,581204,85104,????damages????,-355,2011-12-07 18:32:00,0.00,NaN,United Kingdom
1026494,581205,20893,damages,-55,2011-12-07 18:32:00,0.00,NaN,United Kingdom
1026495,581206,21693,mixed up,-87,2011-12-07 18:34:00,0.00,NaN,United Kingdom
1026496,581207,21688,mixed up,-337,2011-12-07 18:34:00,0.00,NaN,United Kingdom
1026497,581208,72801C,check,-10,2011-12-07 18:35:00,0.00,NaN,United Kingdom
1026499,581210,23395,check,-26,2011-12-07 18:36:00,0.00,NaN,United Kingdom
1026501,581212,22578,lost,-1050,2011-12-07 18:38:00,0.00,NaN,United Kingdom
1026502,581213,22576,check,-30,2011-12-07 18:38:00,0.00,NaN,United Kingdom
1028076,581226,23090,missing,-338,2011-12-08 09:56:00,0.00,NaN,United Kingdom
1030065,581422,23169,smashed,-235,2011-12-08 15:24:00,0.00,NaN,United Kingdom


In [28]:
inventory_adjustments = df[
    (df["Quantity"] < 0) ].copy()

print(f"Inventory Adjustment Records: {len(inventory_adjustments)}")

Inventory Adjustment Records: 760


In [30]:
inventory_adjustments.to_csv(
    "inventory_adjustments.csv",
    index=False
)

In [31]:
returns_df.to_csv(
    "returns_df.csv",
    index=False
)

In [32]:
len(df)

1009657

In [33]:
df = df[
    ~((df["Quantity"] < 0))].copy()

In [35]:
print("Remaining Rows:" ,len(df))

Remaining Rows: 1008897


## Cleaning Decision – Inventory Adjustment Records

A small number of transactions contained negative quantities without cancellation invoice numbers.

Manual inspection showed these records represented internal inventory events such as damaged stock, lost items, stock checks, and warehouse adjustments rather than customer purchases.

These records were preserved in a separate dataset (`inventory_adjustments.csv`) and removed from the primary analytical dataset to ensure that downstream analyses reflect genuine customer transactions.

In [ ]:
## Validation check
(df["Quantity"] < 0).sum()

np.int64(0)

In [38]:
df['Customer ID'].isnull().sum()

np.int64(229402)

In [39]:
zero_price = df[df["Price"] <= 0]

print(f"Zero/Negative Price Records: {len(zero_price)}")

zero_price.head(20)

Zero/Negative Price Records: 984


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4607,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.00,16126.00,United Kingdom
5833,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.00,NaN,United Kingdom
6702,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.00,15658.00,United Kingdom
15906,490727,M,Manual,1,2009-12-07 16:38:00,0.00,17231.00,United Kingdom
18531,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.00,14108.00,United Kingdom
18532,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.00,14108.00,United Kingdom
31624,491971,85042,ANTIQUE LILY FAIRY LIGHTS,1,2009-12-14 18:37:00,0.00,NaN,United Kingdom
32543,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.00,15070.00,United Kingdom
39637,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.00,18071.00,United Kingdom
46602,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.00,14258.00,United Kingdom


In [40]:
zero_price["Description"].value_counts().head(20)

Description
check                              39
found                              28
adjustment                         14
OWL DOORSTOP                       14
POLYESTER FILLER PAD 45x45cm       12
POLYESTER FILLER PAD 40x40cm       10
Found                               9
FRENCH BLUE METAL DOOR SIGN 1       9
?                                   9
amazon                              8
PICNIC BASKET WICKER LARGE          8
RECIPE BOX PANTRY YELLOW DESIGN     8
BOX OF 24 COCKTAIL PARASOLS         8
FRENCH BLUE METAL DOOR SIGN 8       8
RED KITCHEN SCALES                  8
IVORY KITCHEN SCALES                8
WATERING CAN BLUE ELEPHANT          7
CHILDS GARDEN SPADE BLUE            7
FRENCH BLUE METAL DOOR SIGN 4       7
ENAMEL WASH BOWL CREAM              7
Name: count, dtype: int64

In [41]:
zero_price.to_csv(
    "zero_price_transactions.csv",
    index=False
)

In [42]:
df = df[df["Price"] > 0].copy()

In [43]:
len(df)

1007913

In [ ]:
customer_df = df[df["Customer ID"].notna()].copy()

print(customer_df.shape)


(779425, 8)


In [46]:
customer_df.to_csv("customer_df.csv",index=False)

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1007913 entries, 0 to 1033035
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1007913 non-null  object        
 1   StockCode    1007913 non-null  object        
 2   Description  1007913 non-null  object        
 3   Quantity     1007913 non-null  int64         
 4   InvoiceDate  1007913 non-null  datetime64[ns]
 5   Price        1007913 non-null  float64       
 6   Customer ID  779425 non-null   float64       
 7   Country      1007913 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 69.2+ MB


In [48]:
df.to_csv("retail_clean_master.csv", index=False)


# Summary

The following data quality issues were identified and resolved:

- Removed duplicate records.
- Removed overlapping records between yearly datasets.
- Removed records with missing product descriptions.
- Separated cancelled transactions into a dedicated dataset.
- Separated inventory adjustment records into a dedicated dataset.
- Removed transactions with zero or negative prices.
- Preserved transactions with missing Customer IDs in the master dataset while creating a dedicated customer analysis dataset.

The cleaned datasets are now ready for feature engineering, SQL analysis, Power BI dashboards, customer analytics, and machine learning models.